In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)

crop_path = "../data/raw/crop_yield.csv"

df = pd.read_csv(crop_path)
print("Raw shape:", df.shape)
df.head()

In [ ]:
# Basic info and missing values
display(df.info())
df.isna().sum()

In [ ]:
# Clean whitespace in categorical columns
for col in ["crop", "season", "state"]:
    df[col] = df[col].astype(str).str.strip()

# Ensure numeric types
for col in ["year", "area", "production", "fertilizer", "pesticide", "yield"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("After basic cleaning:", df.shape)
df.describe(include="all").T

In [ ]:
# Yield distribution and basic analysis
plt.figure(figsize=(8, 4))
sns.histplot(df["yield"].dropna(), bins=50, kde=True)
plt.title("Distribution of yield")
plt.show()

print("Top crops by mean yield (overall):")
display(
    df.groupby("crop")["yield"].mean().sort_values(ascending=False).head(20)
)

print("Top states by mean yield (overall):")
display(
    df.groupby("state")["yield"].mean().sort_values(ascending=False).head(20)
)

In [ ]:
# Yield trends over time for a few major crops
major_crops = ["Rice", "Wheat", "Maize", "Sugarcane"]
subset = df[df["crop"].isin(major_crops)]

plt.figure(figsize=(10, 6))
sns.lineplot(data=subset, x="year", y="yield", hue="crop", estimator="mean")
plt.title("Mean yield over time for selected crops")
plt.legend()
plt.show()

subset.groupby(["crop", "year"])['yield'].mean().unstack(0).tail(10)

In [ ]:
# Save a cleaned version for modeling
clean_path = "../data/processed/crop_yield_clean.csv"
import os
os.makedirs(os.path.dirname(clean_path), exist_ok=True)
df.to_csv(clean_path, index=False)
print("Saved cleaned crop yield data to", clean_path)